# Problem Set 6 

## Question (a)

 Using the 30 value-weighted industry portfolios, form at the end of each month $t-1$ a portfolio that is long the three industries with the highest month-$t-1$ return (winners, $W_{t-1}$) and short the three industries with the lowest month-$t-1$ return (losers, $L_{t-1}$). Each leg is equal-weighted and the position is held over month $t$:

$$
MOM_t=\frac{1}{3}\sum_{i\in W_{t-1}}R_{i,t}-\frac{1}{3}\sum_{i\in L_{t-1}}R_{i,t}.
$$

We report the time-series mean, standard deviation, $t$-statistic and Sharpe ratio of $MOM_t$:

$$
\overline{MOM}=\frac{1}{T}\sum_{t=1}^{T}MOM_t,\qquad
s_{MOM}=\sqrt{\frac{1}{T-1}\sum_{t=1}^{T}\left(MOM_t-\overline{MOM}\right)^2},\qquad
t=\frac{\overline{MOM}}{s_{MOM}/\sqrt{T}},\qquad
SR=\frac{\overline{MOM}}{s_{MOM}}.
$$

Because $MOM_t$ is the return on a zero-cost (long–short) portfolio, it is already an excess return, so the risk-free rate is not subtracted when computing the Sharpe ratio.

In [1]:
import math
import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

DATA_PATH = "Problem_Set6_2026.xlsx"  
N_LEG = 3                              

In [2]:
xls = pd.ExcelFile(DATA_PATH)
print("Sheets in workbook:", xls.sheet_names)

raw = pd.read_excel(DATA_PATH, sheet_name="30 industries", header=None)
print("\nRaw shape of '30 industries' sheet:", raw.shape)
print("\nFirst 6 rows, first 8 columns (as stored):")
print(raw.iloc[:6, :8].to_string())

Sheets in workbook: ['30 industries', 'Fama-French factors', 'Commodities', '125 Size-Past-return portfolios']

Raw shape of '30 industries' sheet: (1204, 31)

First 6 rows, first 8 columns (as stored):
                                    0      1       2      3      4       5       6       7
0  Industry value-weighted portfolios    NaN     NaN    NaN    NaN     NaN     NaN     NaN
1                                 NaN    NaN     NaN    NaN    NaN     NaN     NaN     NaN
2                                 NaN    NaN     NaN    NaN    NaN     NaN     NaN     NaN
3                                 NaN   Food    Beer  Smoke  Games   Books   Hshld   Clths
4                              192607 0.5300 -5.1900 1.2900 2.9300 10.9700 -0.4900  8.0800
5                              192608 2.6200 27.0300 6.5200 0.5500 10.0100 -3.5800 -2.5000


In [3]:
ind = pd.read_excel(DATA_PATH, sheet_name="30 industries", header=3)
ind = ind.rename(columns={ind.columns[0]: "yyyymm"})

# Drop any fully empty rows (none are expected, but we check explicitly)
n_before = len(ind)
ind = ind.dropna(how="all")
print("Fully empty rows dropped:", n_before - len(ind))

# Dates: YYYYMM integers -> monthly periods
ind["yyyymm"] = ind["yyyymm"].astype(int)
ind.index = pd.PeriodIndex(ind["yyyymm"].astype(str), freq="M")
ind.index.name = "date"
R = ind.drop(columns="yyyymm").astype(float)

industries = list(R.columns)
print("Number of industry columns:", len(industries))
print("Industries:", industries)
assert len(industries) == 30 and len(set(industries)) == 30

print("\nFirst month:", R.index[0], "| Last month:", R.index[-1], "| Months:", len(R))

Fully empty rows dropped: 0
Number of industry columns: 30
Industries: ['Food', 'Beer', 'Smoke', 'Games', 'Books', 'Hshld', 'Clths', 'Hlth', 'Chems', 'Txtls', 'Cnstr', 'Steel', 'FabPr', 'ElcEq', 'Autos', 'Carry', 'Mines', 'Coal', 'Oil', 'Util', 'Telcm', 'Servs', 'BusEq', 'Paper', 'Trans', 'Whlsl', 'Rtail', 'Meals', 'Fin', 'Other']

First month: 1926-07 | Last month: 2026-06 | Months: 1200


In [5]:

gaps = pd.Series([(b - a).n for a, b in zip(R.index[:-1], R.index[1:])]).value_counts()
print("Distribution of month-to-month steps:", dict(gaps))
assert R.index.is_unique and set(gaps.index) == {1}, "Dates are not a contiguous monthly series"

n_nan = int(R.isna().sum().sum())
n_codes = int(R.isin([-99.99, -999]).sum().sum())
print("Missing (NaN) observations:", n_nan)
print("Observations equal to -99.99 or -999:", n_codes)

vals = R.stack()
print(f"\nPooled returns: mean = {vals.mean():.3f}, median = {vals.median():.3f}, "
      f"std = {vals.std():.3f}, min = {vals.min():.2f}, max = {vals.max():.2f}")
print("Share of |returns| > 1:", f"{(vals.abs() > 1).mean():.1%}")

Distribution of month-to-month steps: {1: np.int64(1199)}
Missing (NaN) observations: 0
Observations equal to -99.99 or -999: 0

Pooled returns: mean = 1.025, median = 1.070, std = 7.066, min = -50.64, max = 125.43
Share of |returns| > 1: 84.6%


**Data notes.**

* The `30 industries` sheet has three title/blank rows; industry names are in the fourth row (`header=3`), and dates are `YYYYMM` integers in the first column.
* The sheet contains 30 industries at a monthly frequency from 1926:07 to 2026:06 (1,200 months), with no gaps or duplicate dates.
* There are no missing values and no `-99.99`/`-999` missing-data codes in this sheet, so **no observations are filled or excluded**.
* Returns are in **percent per month**: the pooled median is about 1, 84.6% of observations exceed 1 in absolute value, and the extremes (−50.64 and +125.43) are only plausible as percentages. All results below are therefore in percent per month.
* The assignment refers to `Problem_Set6_2026.xls`; the file supplied is `Problem_Set6_2026.xlsx`. The contents match the description (four sheets).
* The T-bill rate is not needed, because the strategy is a zero-cost long–short portfolio.

**Portfolio construction.**

* **Timing (no look-ahead).** For each holding month $t$, industries are ranked using only month-$t-1$ returns (`R.shift(1)`), and the resulting weights are applied to month-$t$ returns. Because 1926:07 is used only for ranking, the holding sample is 1926:08–2026:06 ($T$ = 1,199), the maximum the data allow.
* **Weights.** Each winner receives $+1/3$ and each loser $-1/3$, so the long leg sums to $+1$, the short leg to $-1$, and the net investment is zero.
* **Ties.** Returns are reported to two decimals, so industries can tie at the 3rd/4th (or 27th/28th) rank. The main rule, `keep="first"`, breaks ties by column order; its effect is assessed below.

In [6]:
formation = R.shift(1)                 # month t-1 returns, aligned with holding month t
hold_dates = R.index[1:]               # first holding month needs one prior month

w_long  = pd.DataFrame(0.0, index=hold_dates, columns=industries)
w_short = pd.DataFrame(0.0, index=hold_dates, columns=industries)
records = []

for t in hold_dates:
    f = formation.loc[t]               # information set: returns of month t-1 only
    assert f.notna().all()             # holds here; would require explicit handling otherwise
    winners = f.nlargest(N_LEG, keep="first").index
    losers  = f.nsmallest(N_LEG, keep="first").index

    w_long.loc[t, winners] = 1.0 / N_LEG
    w_short.loc[t, losers] = 1.0 / N_LEG

    r_w = R.loc[t, winners].mean()     # equal-weighted winner return in month t
    r_l = R.loc[t, losers].mean()      # equal-weighted loser  return in month t
    records.append({"date": t, "formation_month": t - 1,
                    "winners": list(winners), "losers": list(losers),
                    "R_W": r_w, "R_L": r_l, "MOM": r_w - r_l})

mom = pd.DataFrame(records).set_index("date")
print("Holding months:", mom.index[0], "to", mom.index[-1], "| T =", len(mom))
mom[["formation_month", "winners", "losers", "R_W", "R_L", "MOM"]].head(3)

Holding months: 1926-08 to 2026-06 | T = 1199


,formation_month,winners,losers,R_W,R_L,MOM
date,,,,,,
1926-08,1926-07,"[Autos, Books, Servs]","[Whlsl, Beer, Oil]",5.4267,12.0200,-6.5933
1926-09,1926-08,"[Beer, Books, Txtls]","[Hshld, Clths, Paper]",1.7833,-1.7733,3.5567
1926-10,1926-09,"[Games, Chems, Autos]","[Whlsl, Paper, Oil]",-5.8733,-7.1667,1.2933


In [7]:
x = mom["MOM"]
T = x.count()
mean_mom = x.mean()
sd_mom = x.std(ddof=1)
se_mom = sd_mom / np.sqrt(T)
t_stat = mean_mom / se_mom
sharpe = mean_mom / sd_mom

# Two-sided p-value. With T-1 = 1198 degrees of freedom the t distribution is
# numerically indistinguishable from the standard normal, so we use the normal CDF.
p_value = math.erfc(abs(t_stat) / math.sqrt(2))

results = pd.DataFrame({
    "Value": [f"{mean_mom:.4f}%", f"{sd_mom:.4f}%", f"{t_stat:.4f}", f"{sharpe:.4f}",
              f"{mom.index[0].strftime('%Y:%m')} – {mom.index[-1].strftime('%Y:%m')}", f"{T}"]
}, index=["Average monthly momentum return", "Monthly standard deviation",
          "t-statistic (H0: mean = 0)", "Monthly Sharpe ratio",
          "Sample period (holding months)", "Number of observations"])
results.index.name = "Statistic"

print(f"Two-sided p-value: {p_value:.2e}  |  5% critical value ≈ 1.96")
print(f"Mean winner-leg return: {mom['R_W'].mean():.4f}%  |  Mean loser-leg return: {mom['R_L'].mean():.4f}%")
results

Two-sided p-value: 4.17e-06  |  5% critical value ≈ 1.96
Mean winner-leg return: 1.3399%  |  Mean loser-leg return: 0.6019%


,Value
Statistic,
Average monthly momentum return,0.7380%
Monthly standard deviation,5.5523%
t-statistic (H0: mean = 0),4.6027
Monthly Sharpe ratio,0.1329
Sample period (holding months),1926:08 – 2026:06
Number of observations,1199


**Verification.** The cells below check the construction independently of the main loop:

1. Winner/loser selection and holding-month returns are recomputed by hand for 1926:08 and 2026:06.
2. Every portfolio is formed in month $t-1$ and held in month $t$; selections are the top/bottom three on $t-1$ returns in every month; a look-ahead placebo (ranking on month-$t$ returns) is shown for contrast.
3. Each leg has exactly three weights of $1/3$ summing to one, the legs never overlap, and $MOM_t$ equals both $R_{W,t}-R_{L,t}$ and the weight vector applied to month-$t$ returns.
4. The summary statistics are recomputed with plain NumPy formulas.

In [8]:
def manual_check(hold_month: str):
    t = pd.Period(hold_month, freq="M")
    t_minus_1 = t - 1
    prev_row = R.loc[t_minus_1].to_dict()               # formation month, read directly
    ranked = sorted(prev_row.items(), key=lambda kv: kv[1], reverse=True)
    W = [k for k, _ in ranked[:3]]
    L = [k for k, _ in ranked[-3:]]
    rw = sum(R.loc[t, k] for k in W) / 3
    rl = sum(R.loc[t, k] for k in L) / 3

    print(f"Holding month {t} (ranked on {t_minus_1})")
    tbl = pd.DataFrame({
        "leg": ["Winner"] * 3 + ["Loser"] * 3,
        "industry": W + L,
        f"R({t_minus_1})": [prev_row[k] for k in W + L],
        f"R({t})": [R.loc[t, k] for k in W + L],
    })
    print(tbl.to_string(index=False))
    print(f"  4th-highest formation return: {ranked[3][1]:.2f} ({ranked[3][0]}); "
          f"4th-lowest: {ranked[-4][1]:.2f} ({ranked[-4][0]})")
    print(f"  Manual:   R_W = {rw:.4f}, R_L = {rl:.4f}, MOM = {rw - rl:.4f}")
    print(f"  Pipeline: R_W = {mom.loc[t, 'R_W']:.4f}, R_L = {mom.loc[t, 'R_L']:.4f}, "
          f"MOM = {mom.loc[t, 'MOM']:.4f}")
    assert set(W) == set(mom.loc[t, "winners"]) and set(L) == set(mom.loc[t, "losers"])
    assert np.isclose(rw - rl, mom.loc[t, "MOM"])
    print("  -> match\n")

manual_check("1926-08")
manual_check("2026-06")

Holding month 1926-08 (ranked on 1926-07)
   leg industry  R(1926-07)  R(1926-08)
Winner    Autos     14.5600      4.2500
Winner    Books     10.9700     10.0100
Winner    Servs      9.3200      2.0200
 Loser      Oil     -1.3500      3.6400
 Loser     Beer     -5.1900     27.0300
 Loser    Whlsl    -23.7900      5.3900
  4th-highest formation return: 8.16 (Chems); 4th-lowest: -0.49 (Hshld)
  Manual:   R_W = 5.4267, R_L = 12.0200, MOM = -6.5933
  Pipeline: R_W = 5.4267, R_L = 12.0200, MOM = -6.5933
  -> match

Holding month 2026-06 (ranked on 2026-05)
   leg industry  R(2026-05)  R(2026-06)
Winner    BusEq     14.3500     -0.9700
Winner    Autos     12.7800     -3.4400
Winner    Steel     11.3600     -5.0900
 Loser     Util     -4.9700      2.8000
 Loser      Oil     -6.1600     -5.6900
 Loser     Beer     -7.4500     -2.0000
  4th-highest formation return: 7.68 (Servs); 4th-lowest: -4.35 (Games)
  Manual:   R_W = -3.1667, R_L = -1.6300, MOM = -1.5367
  Pipeline: R_W = -3.1667, R_L = -

In [11]:

lag = pd.Series([(t - f).n for t, f in zip(mom.index, mom["formation_month"])])
assert (lag == 1).all()
print("Formation month = holding month - 1 for all", len(mom), "observations")


ok = True
for t in mom.index:
    f = R.loc[t - 1]
    W, L = mom.loc[t, "winners"], mom.loc[t, "losers"]
    ok &= f[W].min() >= f.drop(W).max()
    ok &= f[L].max() <= f.drop(L).min()
assert ok
print("Winners/losers are the top/bottom 3 on month t-1 returns in every month")

look_ahead = []
for t in mom.index:
    g = R.loc[t]
    look_ahead.append(g.nlargest(3).mean() - g.nsmallest(3).mean())
look_ahead = pd.Series(look_ahead, index=mom.index)
print(f"Months where correct and look-ahead portfolios coincide: "
      f"{(np.isclose(look_ahead, mom['MOM'])).sum()} of {len(mom)}")
print(f"Mean MOM (correct timing): {mom['MOM'].mean():.4f}%  vs  "
      f"mean with look-ahead: {look_ahead.mean():.4f}%")

Formation month = holding month - 1 for all 1199 observations
Winners/losers are the top/bottom 3 on month t-1 returns in every month
Months where correct and look-ahead portfolios coincide: 0 of 1199
Mean MOM (correct timing): 0.7380%  vs  mean with look-ahead: 13.4810%


In [ ]:

assert np.allclose(w_long.sum(axis=1), 1.0) and np.allclose(w_short.sum(axis=1), 1.0)
assert ((w_long > 0).sum(axis=1) == 3).all() and ((w_short > 0).sum(axis=1) == 3).all()
wl, ws = w_long.to_numpy(), w_short.to_numpy()
assert np.allclose(wl[wl > 0], 1/3) and np.allclose(ws[ws > 0], 1/3)
assert ((w_long > 0) & (w_short > 0)).sum().sum() == 0
net = (w_long - w_short).sum(axis=1)
print("Long-leg weight sums:  min = %.6f, max = %.6f" % (w_long.sum(axis=1).min(), w_long.sum(axis=1).max()))
print("Short-leg weight sums: min = %.6f, max = %.6f" % (w_short.sum(axis=1).min(), w_short.sum(axis=1).max()))
print("Net (long - short) weight: max |net| = %.2e  -> zero-cost portfolio" % net.abs().max())

# MOM equals winners minus losers
diff1 = (mom["MOM"] - (mom["R_W"] - mom["R_L"])).abs().max()
# ... and equals the weight vector applied to month-t returns (independent matrix route)
mom_matrix = ((w_long - w_short) * R.loc[hold_dates]).sum(axis=1)
diff2 = (mom["MOM"] - mom_matrix).abs().max()
print(f"max |MOM - (R_W - R_L)|        = {diff1:.2e}")
print(f"max |MOM - (w_L - w_S)'R_t|    = {diff2:.2e}")
assert diff1 < 1e-10 and diff2 < 1e-10

Long-leg weight sums:  min = 1.000000, max = 1.000000
Short-leg weight sums: min = 1.000000, max = 1.000000
Net (long - short) weight: max |net| = 1.11e-16  -> zero-cost portfolio
max |MOM - (R_W - R_L)|        = 0.00e+00
max |MOM - (w_L - w_S)'R_t|    = 7.11e-15


In [15]:
v = mom["MOM"].to_numpy()
T_chk = v.size
mean_chk = v.sum() / T_chk
sd_chk = np.sqrt(((v - mean_chk) ** 2).sum() / (T_chk - 1))   # ddof = 1
t_chk = mean_chk / (sd_chk / np.sqrt(T_chk))
sr_chk = mean_chk / sd_chk

check = pd.DataFrame({
    "Pipeline": [mean_mom, sd_mom, t_stat, sharpe, T],
    "Independent": [mean_chk, sd_chk, t_chk, sr_chk, T_chk],
}, index=["Mean (%)", "Std. dev. (%)", "t-statistic", "Sharpe ratio", "T"])
check["Abs. difference"] = (check["Pipeline"] - check["Independent"]).abs()
assert np.allclose(check["Pipeline"], check["Independent"])
assert np.isclose(t_chk, sr_chk * np.sqrt(T_chk))   # identity: t = SR * sqrt(T)
print("Identity check t = SR x sqrt(T):", f"{sr_chk * np.sqrt(T_chk):.4f}")
check

Identity check t = SR x sqrt(T): 4.6027


,Pipeline,Independent,Abs. difference
Mean (%),0.7380,0.7380,0.0000
Std. dev. (%),5.5523,5.5523,0.0000
t-statistic,4.6027,4.6027,0.0000
Sharpe ratio,0.1329,0.1329,0.0000
T,"1,199.0000","1,199.0000",0.0000


**Tie sensitivity.** In the months where two industries share the same formation return at the selection cut-off, the composition of the portfolio depends on the tie-breaking rule. The cell below counts these months and recomputes the statistics under `keep="last"` and under a rule that splits the remaining weight equally among all tied industries (so each leg still sums to one).

In [16]:
def leg_weights(f, top):
    """Equal weights on the 3 extreme industries; industries tied at the cut-off share the leftover weight."""
    s = f.sort_values(ascending=not top)
    cut = s.iloc[2]
    strictly_in = s[s > cut] if top else s[s < cut]
    tied = s[np.isclose(s, cut)]
    w = pd.Series(0.0, index=f.index)
    w[strictly_in.index] = 1 / 3
    w[tied.index] = (1 - len(strictly_in) / 3) / len(tied)
    return w

def mom_series(rule):
    out = []
    for t in hold_dates:
        f, r = formation.loc[t], R.loc[t]
        if rule in ("first", "last"):
            out.append(r[f.nlargest(3, keep=rule).index].mean() - r[f.nsmallest(3, keep=rule).index].mean())
        else:
            out.append((leg_weights(f, True) * r).sum() - (leg_weights(f, False) * r).sum())
    return pd.Series(out, index=hold_dates)

def has_tie(f):
    d, a = f.sort_values(ascending=False), f.sort_values()
    return np.isclose(d.iloc[2], d.iloc[3]) or np.isclose(a.iloc[2], a.iloc[3])

tie_months = [t for t in hold_dates if has_tie(formation.loc[t])]
print(f"Months with a tie at the winner or loser cut-off: {len(tie_months)} of {len(hold_dates)}")
print("Affected holding months:", ", ".join(str(t) for t in tie_months))

rows = {}
for rule, label in [("first", "keep='first' (main)"), ("last", "keep='last'"), ("split", "split tied weight")]:
    s = mom_series(rule)
    rows[label] = {"Mean (%)": s.mean(), "Std. dev. (%)": s.std(ddof=1),
                   "t-statistic": s.mean() / (s.std(ddof=1) / np.sqrt(s.size)),
                   "Sharpe ratio": s.mean() / s.std(ddof=1)}
assert np.allclose(mom_series("first"), mom["MOM"])
pd.DataFrame(rows).T

Months with a tie at the winner or loser cut-off: 14 of 1199
Affected holding months: 1936-09, 1940-09, 1943-10, 1948-10, 1952-03, 1960-02, 1968-12, 1971-01, 1971-10, 1972-12, 1984-05, 1994-01, 2017-08, 2017-10


,Mean (%),Std. dev. (%),t-statistic,Sharpe ratio
keep='first' (main),0.7380,5.5523,4.6027,0.1329
keep='last',0.7366,5.5517,4.5941,0.1327
split tied weight,0.7373,5.5519,4.5985,0.1328


**Table 1. 1-month, 1-month industry momentum strategy (30 value-weighted industries).**

| Statistic | Value |
|---|---|
| Average monthly momentum return | 0.7380% |
| Monthly standard deviation | 5.5523% |
| t-statistic | 4.60 |
| Monthly Sharpe ratio | 0.1329 |
| Sample period | 1926:08 – 2026:06 (holding months) |
| Number of observations | 1,199 |

*Notes:* At the end of each month $t-1$, the 30 value-weighted industry portfolios are ranked by their month-$t-1$ return. The strategy goes long the top three (winners) and short the bottom three (losers), with equal weights of 1/3 in each leg, and holds the position over month $t$. Returns are in percent per month. The standard deviation is the sample standard deviation ($ddof=1$). The $t$-statistic tests $H_0:\ E[MOM_t]=0$ and is computed as $\overline{MOM}/(s_{MOM}/\sqrt{T})$. The Sharpe ratio is $\overline{MOM}/s_{MOM}$; no risk-free rate is subtracted because the strategy is zero-cost. No observations were missing or excluded. In 14 months, two industries tie at the selection cut-off; the tie-sensitivity table above shows that the results do not depend on how these ties are broken (mean between 0.7366% and 0.7380%).

**Interpretation.**

*Sign of the return.* The strategy earns a positive average return of 0.7380% per month. Last month's three best industries earn 1.3399% on average in the following month, while last month's three worst earn 0.6019%. Industries that have recently done well therefore continue to outperform recent losers over the next month, consistent with the short-horizon industry momentum documented by Moskowitz and Grinblatt (1999), and in contrast to the one-month reversal usually found for individual stocks.

*Statistical significance.* The $t$-statistic of 4.60 exceeds the two-sided 5% critical value of about 1.96 (two-sided $p$-value of about $4.2\times10^{-6}$), so we reject the null hypothesis that the mean momentum return is zero at the 5% level. The test assumes serially uncorrelated, homoskedastic monthly returns; a heteroskedasticity- and autocorrelation-consistent standard error could change the exact $t$-value.

*Risk-adjusted performance.* The monthly Sharpe ratio is 0.1329: each percentage point of monthly volatility earns about 0.13 percentage points of average return. The strategy is very volatile (5.5523% per month against a mean of 0.7380%) because it holds only three industries per leg and turns over fully each month. The large $t$-statistic reflects the long sample ($t = SR\times\sqrt{T}$) rather than a high Sharpe ratio. These are gross returns before transaction costs, which would lower the net Sharpe ratio for a strategy with this much turnover.

# Question (b)




**Framework.** The Lo–MacKinlay (1990) / Moskowitz–Grinblatt (1999) decomposition applies to the *weighted relative-strength strategy* (WRSS). This strategy holds industry $i$ in month $t$ with weight $w_{i,t}=\frac{1}{N}\left(R_{i,t-1}-\bar R_{t-1}\right)$, where $\bar R_{t-1}$ is the equal-weighted cross-sectional mean and $N=30$. It is the linear, zero-cost version of the 1-month, 1-month strategy in (a). Write each industry's return with the market model, $R_{i,t}=\alpha_i+\beta_i R_{m,t}+e_{i,t}$, and let $\tilde e_{i,t}=e_{i,t}-\bar e_t$. Then

$$
E[\pi_t]=\underbrace{\sigma^2_\mu}_{(1)}+\underbrace{\sigma^2_\beta\,\mathrm{Cov}(R_{m,t},R_{m,t-1})}_{(2)}+\underbrace{\tfrac{1}{N}\textstyle\sum_i \mathrm{Cov}(\tilde e_{i,t},\tilde e_{i,t-1})}_{(3)}
\;+\;\underbrace{\tfrac{1}{N}\textstyle\sum_i(\beta_i-\bar\beta)\left[\mathrm{Cov}(R_{m,t-1},\tilde e_{i,t})+\mathrm{Cov}(\tilde e_{i,t-1},R_{m,t})\right]}_{\text{lead–lag market/residual terms, assumed}\ \approx 0}
$$

* $\sigma^2_\mu$ is the cross-sectional variance of the 30 industry sample means.
* $\sigma^2_\beta$ is the cross-sectional variance of the market betas.
* Term (3) is the average cross-autocovariance of the residuals. Because the residuals are demeaned cross-sectionally, it equals $\frac{1}{N}\sum_i\gamma_{ii}-\frac{1}{N^2}\sum_i\sum_j\gamma_{ij}$, with $\gamma_{ij}=\mathrm{Cov}(e_{i,t-1},e_{j,t})$. It is therefore the industries' own residual autocovariances net of the average lead–lag covariance across all pairs of industries.

**Implementation choices.**

* The market return is the CRSP value-weighted market from the Fama–French sheet, $R_m=$ `Mkt-RF` $+$ `RF`. This puts it in the same total-return units as the industry returns.
* Betas come from full-sample time-series OLS of the market model, 1926:07–2026:06.
* Cross-sectional variances use $1/N$, and autocovariances use the $T-1$ overlapping $(t-1,t)$ pairs (ddof = 0). With these choices the identity holds exactly in sample.
* Units are %$^2$ per month, because both the WRSS weight and the return are in percent.

In [18]:
ff = pd.read_excel(DATA_PATH, sheet_name="Fama-French factors", header=2)
ff = ff.rename(columns={ff.columns[0]: "yyyymm"}).dropna(how="all")
ff.index = pd.PeriodIndex(ff["yyyymm"].astype(int).astype(str), freq="M")

assert ff.index.equals(R.index), "Factor and industry dates do not align"
assert not ff[["Mkt-RF", "RF"]].isin([-99.99, -999]).any().any() and ff[["Mkt-RF", "RF"]].notna().all().all()
Rm = (ff["Mkt-RF"] + ff["RF"]).astype(float)          # total market return, % per month
print("Market series:", Rm.index[0], "to", Rm.index[-1], "| T =", len(Rm), "| no missing or -999 codes")

Market series: 1926-07 to 2026-06 | T = 1200 | no missing or -999 codes


In [19]:
N_IND = R.shape[1]

# Market-model OLS for each industry: R_i = a_i + b_i R_m + e_i
X = np.column_stack([np.ones(len(Rm)), Rm.to_numpy()])
coef, *_ = np.linalg.lstsq(X, R.to_numpy(), rcond=None)
beta = pd.Series(coef[1], index=industries, name="beta")
E = pd.DataFrame(R.to_numpy() - X @ coef, index=R.index, columns=industries)   # residuals
assert np.allclose(E.mean(), 0) and np.allclose(E.T @ Rm, 0, atol=1e-6)       # OLS orthogonality

def lag_cov(a, b):
    """Cov(a_{t-1}, b_t) over the T-1 overlapping pairs (ddof = 0)."""
    a, b = np.asarray(a)[:-1], np.asarray(b)[1:]
    return np.mean((a - a.mean()) * (b - b.mean()))

E_tilde = E.sub(E.mean(axis=1), axis=0)                 # cross-sectionally demeaned residuals

var_mu   = R.mean().var(ddof=0)                         # (1) cross-sectional variance of means
var_beta = beta.var(ddof=0)
acov_m   = lag_cov(Rm, Rm)                              # first-order autocovariance of market
comp1 = var_mu
comp2 = var_beta * acov_m                               # (2)
comp3 = np.mean([lag_cov(E_tilde[c], E_tilde[c]) for c in industries])   # (3)

# Split of (3): own autocovariances minus average lead-lag covariance across all pairs
own   = np.mean([lag_cov(E[c], E[c]) for c in industries])
cross = np.mean([[lag_cov(E[i], E[j]) for j in industries] for i in industries])
assert np.isclose(comp3, own - cross)

print(f"Betas: mean = {beta.mean():.3f}, min = {beta.min():.2f} ({beta.idxmin()}), max = {beta.max():.2f} ({beta.idxmax()})")
print(f"sigma^2(beta) = {var_beta:.4f} | Cov(Rm_t, Rm_t-1) = {acov_m:.4f} (%^2), first-order autocorr = {acov_m / Rm.var(ddof=0):.3f}")
print(f"Term (3) = own residual autocov {own:.4f} - avg lead-lag cross-cov {cross:.4f} = {comp3:.4f}")

Betas: mean = 1.027, min = 0.63 (Smoke), max = 1.39 (Games)
sigma^2(beta) = 0.0418 | Cov(Rm_t, Rm_t-1) = 2.4998 (%^2), first-order autocorr = 0.089
Term (3) = own residual autocov 0.5710 - avg lead-lag cross-cov 0.1440 = 0.4270


In [20]:
# WRSS profits: weights (1/N)(R_{i,t-1} - Rbar_{t-1}) applied to month-t returns
dev = R.sub(R.mean(axis=1), axis=0)
pi = ((dev.shift(1) * R).sum(axis=1) / N_IND).loc[hold_dates]

# Lead-lag market/residual terms (zero by assumption in the decomposition)
b_dev = beta - beta.mean()
comp4 = np.mean([b_dev[c] * (lag_cov(Rm, E_tilde[c]) + lag_cov(E_tilde[c], Rm)) for c in industries])

# Exact sample identity (dispersion term uses the means of the t-1 and t sub-samples)
disp_exact = np.mean(dev.to_numpy()[:-1].mean(axis=0) * dev.to_numpy()[1:].mean(axis=0))
assert np.isclose(pi.mean(), disp_exact + comp2 + comp3 + comp4)

three = comp1 + comp2 + comp3
decomp = pd.DataFrame({
    "Value (%^2/month)": [comp1, comp2, comp3, three, comp4, pi.mean()],
    "Share of sum of (1)-(3)": [comp1 / three, comp2 / three, comp3 / three, 1.0, np.nan, np.nan],
    "Share of WRSS profit": [comp1 / pi.mean(), comp2 / pi.mean(), comp3 / pi.mean(), three / pi.mean(),
                             comp4 / pi.mean(), 1.0],
}, index=["(1) Cross-sectional variance of mean returns",
          "(2) Var(beta) x market autocovariance",
          "(3) Avg. residual cross-autocovariance",
          "Sum of (1)-(3)",
          "Lead-lag market/residual terms (omitted)",
          "Mean WRSS momentum profit"])

print(f"Identity check: mean WRSS profit {pi.mean():.4f} = components + omitted terms "
      f"{disp_exact + comp2 + comp3 + comp4:.4f}")
print(f"Correlation between WRSS profit and the top-3/bottom-3 MOM series of (a): {pi.corr(mom['MOM']):.3f}")
decomp

Identity check: mean WRSS profit 0.7071 = components + omitted terms 0.7071
Correlation between WRSS profit and the top-3/bottom-3 MOM series of (a): 0.662


,Value (%^2/month),Share of sum of (1)-(3),Share of WRSS profit
(1) Cross-sectional variance of mean returns,0.0153,0.0279,0.0216
(2) Var(beta) x market autocovariance,0.1044,0.1910,0.1476
(3) Avg. residual cross-autocovariance,0.4270,0.7811,0.6039
Sum of (1)-(3),0.5467,1.0000,0.7731
Lead-lag market/residual terms (omitted),0.1605,NaN,0.2270
Mean WRSS momentum profit,0.7071,NaN,1.0000


**Table 2. Decomposition of 1-month, 1-month industry momentum profits (1926:08–2026:06).**

| Component | Value (%²/month) | Share of (1)+(2)+(3) |
|---|---|---|
| (1) Cross-sectional variance of industry mean returns | 0.0153 | 2.8% |
| (2) $\sigma^2_\beta \times$ market autocovariance | 0.1044 | 19.1% |
| (3) Average residual cross-autocovariance | 0.4270 | 78.1% |
| Sum of (1)–(3) | 0.5467 | 100% |
| Mean WRSS momentum profit | 0.7071 | — |

*Notes:* Industry betas are full-sample market-model estimates against the CRSP value-weighted market. $\sigma^2_\beta=$ 0.0418 and $\mathrm{Cov}(R_{m,t},R_{m,t-1})=$ 2.4998 %², a first-order autocorrelation of 0.089. Term (3) is the industries' own residual autocovariance (0.5710) minus the average lead–lag residual covariance across all industry pairs (0.1440). The WRSS profit uses weights $\frac{1}{N}(R_{i,t-1}-\bar R_{t-1})$.

**Answer.**

* **Does the sum approximate the profit?** The three components sum to 0.5467 %², which is 77.3% of the mean WRSS profit (0.7071 %²). The remaining 0.1605 %² comes from the lead–lag covariances between the market and industry residuals, which the decomposition assumes are zero. Once these are added, the sample identity holds exactly. The WRSS is the linear counterpart of the top-3-minus-bottom-3 portfolio in (a), and the two return series have a correlation of 0.662. The decomposition therefore describes the source of the profits in (a), but not its exact size. The WRSS profit is in %², while $\overline{MOM}=$ 0.7380% is in percent. The two numbers happen to be close, but they cannot be compared directly.
* **Which component contributes most?** **Component (3), the residual cross-autocovariance**, accounts for 78.1% of the three-component total.
  * Component (2) contributes 19.1%. The market's autocorrelation is positive but small, and betas vary little across industries.
  * Component (1) contributes only 2.8%, so the profit is *not* mainly the result of winners being industries with permanently higher mean returns.
  * Within (3), the positive own autocovariance of industry-specific returns (0.5710) dominates. Cross-serial covariances between industries partly offset it (0.1440).

  Industry momentum at the one-month horizon therefore reflects continuation of the industry-specific components of returns, rather than dispersion in expected returns or the market factor.

In [24]:
from IPython.display import display

# Check sample alignment and the implemented component formulas.
assert R.index.equals(Rm.index)
assert mom.index.equals(pi.index)
assert mom.index.equals(R.index[1:])
assert np.isclose(comp1, R.mean().var(ddof=0))
assert np.isclose(comp2, beta.var(ddof=0) * lag_cov(Rm, Rm))
assert np.isclose(comp3, own - cross)

comparison_sum = comp1 + comp2 + comp3
comparison_mean = mom["MOM"].mean()
comparison_difference = comparison_sum - comparison_mean

comparison = pd.DataFrame({
    "Value": [
        comp1, comp2, comp3, comparison_sum,
        comparison_mean, comparison_difference
    ],
    "Units / interpretation": [
        "%²/month", "%²/month", "%²/month", "%²/month",
        "%/month",
        "Numerical subtraction only: incompatible units"
    ]
}, index=[
    "Component 1: dispersion in industry mean returns",
    "Component 2: beta dispersion × market autocovariance",
    "Component 3: residual serial covariance",
    "Sum of three components",
    "Part (a): average monthly momentum return",
    "Difference: sum − Part (a)"
])

display(comparison.style.format({"Value": "{:.4f}"}))

# Investigate the gap within WRSS, where units ARE comparable.
# Pair-specific means give the exact finite-sample dispersion term.
comparison_endpoint_adjustment = disp_exact - comp1
comparison_wrss_gap = pi.mean() - comparison_sum

assert np.isclose(
    comparison_wrss_gap,
    comp4 + comparison_endpoint_adjustment,
    rtol=1e-9, atol=1e-10
)

print(f"Aligned holding sample: {mom.index[0]}–{mom.index[-1]}, "
      f"{len(mom)} months")
print(f"Component 3 share of three-component sum: "
      f"{comp3 / comparison_sum:.1%}")
print(f"Mean WRSS profit: {pi.mean():.6f} %²/month")
print(f"WRSS minus three-component sum: {comparison_wrss_gap:.6f} %²/month")
print(f"  Omitted market/residual lead–lag terms: {comp4:.6f}")
print(f"  Endpoint-mean adjustment: {comparison_endpoint_adjustment:.6f}")

,Value,Units / interpretation
Component 1: dispersion in industry mean returns,0.0153,%²/month
Component 2: beta dispersion × market autocovariance,0.1044,%²/month
Component 3: residual serial covariance,0.4270,%²/month
Sum of three components,0.5467,%²/month
Part (a): average monthly momentum return,0.7380,%/month
Difference: sum − Part (a),-0.1914,Numerical subtraction only: incompatible units


Aligned holding sample: 1926-08–2026-06, 1199 months
Component 3 share of three-component sum: 78.1%
Mean WRSS profit: 0.707137 %²/month
WRSS minus three-component sum: 0.160474 %²/month
  Omitted market/residual lead–lag terms: 0.160548
  Endpoint-mean adjustment: -0.000074


| Statistic | Value | Units |
|---|---:|---|
| Component 1: dispersion in industry mean returns | 0.0153 | %²/month |
| Component 2: beta dispersion × market autocovariance | 0.1044 | %²/month |
| Component 3: residual serial covariance | 0.4270 | %²/month |
| Sum of components | 0.5467 | %²/month |
| Part (a): average momentum return | 0.7380 | %/month |
| Sum − Part (a) | −0.1914 | Numerical difference only; units differ |

**Comparison.** All three components and Part (a)'s average return are
positive. Component 3 dominates, contributing 78.1% of the decomposition
sum, compared with 19.1% for Component 2 and 2.8% for Component 1.
Numerically, the sum is about 26% below Part (a), so agreement is not close;
however, this percentage is not an economically meaningful return comparison.

**Checks and discrepancy.** Both strategies use the same 1,199 holding
months, August 1926–June 2026. Part (a) assigns weights of +1/3 and −1/3
to the three winners and losers. Part (b) uses all industries with weights
proportional to lagged returns minus their cross-sectional mean, without
normalizing each leg to one. Because those weights use percentage-valued
returns, its reported profit is in %², whereas Part (a) is in percent.
Thus, the decomposition is not an exact accounting of Part (a)'s return.

Even against its own WRSS benchmark (0.7071 %²/month), the three-component
sum explains only 77.3%. The remaining gap is accounted for by omitted
market–residual lead–lag covariances (0.1605 %²/month), plus a small
endpoint-mean adjustment. Contemporaneous OLS orthogonality does not make
these lead–lag covariances zero.

The decomposition favors residual return continuation over mean dispersion
as the main source of WRSS profits. It does not establish investor
under-reaction; that interpretation requires additional evidence about
information arrival and delayed price responses, while assessing alternative
risk-based explanations.

### AI Integration Discussion

(i) Distinguishing the components. The AI correctly separated cross-sectional dispersion in industry mean returns, market autocovariance multiplied by the cross-sectional variance of betas, and residual serial covariance. Component 3 uses market-model residuals rather than raw returns and includes own-industry autocovariances net of average cross-industry lead–lag covariances.

(ii) Verification and comparison. We checked that the market and industry dates align and that both strategies use the same 1,199 holding months. We also verified that Component 3 equals own-residual autocovariance minus average cross-industry covariance, and that adding the omitted market–residual lead–lag terms and endpoint adjustment reproduces the WRSS profit.

The three components sum to 0.5467 %²/month, while Part (a)’s average return is 0.7380%/month. Both are positive, but their magnitudes are not directly comparable: the decomposition uses unnormalized, return-proportional weights across all industries, whereas Part (a) assigns equal weights to the top and bottom three. Even relative to the WRSS profit of 0.7071 %²/month, the sum explains only 77.3%. The remaining gap primarily reflects omitted market–residual lead–lag covariances. The AI’s claim that the three terms alone give an exact sample identity therefore needs qualification.

(iii) Interpretation. Component 3 contributes 78.1% of the three-component sum, compared with 19.1% for Component 2 and 2.8% for Component 1. Thus, residual serial dependence, rather than persistent differences in industry mean returns, dominates this decomposition. This supports a return-continuation interpretation but does not establish investor under-reaction. That would require evidence linking predictable subsequent returns to delayed responses to identifiable information, while evaluating alternative explanations such as time-varying risk premia.